# P82 — Predecir buenas probabilidades con aprendizaje supervisado

## 1. Título y paper

**Paper:** *Predicting Good Probabilities with Supervised Learning*  
**Autoría:** Alexandru Niculescu-Mizil, Rich Caruana  
**Año y venue:** 2005 · ICML '05, 625–632  
**Nivel:** L3 · **Motor:** `calibracion`  
**Ficha completa:** [`P82_calibracion`](../../papers/foundational/P82_calibracion/README.md)

**Hito:** Separa dos cosas que se confundían: ordenar bien los ejemplos y estimar bien la probabilidad de cada uno.

- [doi:10.1145/1102351.1102430](https://doi.org/10.1145/1102351.1102430)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las salidas de un clasificador se usan como probabilidades para decidir con umbrales de coste o para combinarlas con otras. Pero un modelo puede tener un AUC excelente y probabilidades sistemáticamente sesgadas, y nadie lo estaba midiendo.
2. Ejecutar una implementación mínima de la propuesta: Medir la calibración con diagramas de fiabilidad y puntuaciones propias, caracterizar cómo se descalibra cada familia de modelos, y corregirla con escalado de Platt o regresión isotónica sin alterar el orden.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Platt (1999), salidas probabilísticas para SVM
- P80


## 4. Intuición

Un modelo dice «0,9». ¿Significa que de cada diez casos así, nueve ocurren? No necesariamente. Un modelo puede ordenar perfectamente y estimar fatal, y las dos cosas se miden con métricas distintas que casi nunca se reportan juntas.


## 5. Concepto mínimo

```text
Ordenar bien   → AUC alto
Estimar bien   → calibración: de los casos con p̂ ≈ 0,9, ocurre el 90 %

Diagrama de fiabilidad: p̂ media por tramo  frente a  frecuencia observada
Brier = media((p̂ − y)²)     ← castiga orden Y calibración a la vez
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('calibracion', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué AUC tiene el modelo?
2. ¿Coinciden sus probabilidades con las frecuencias observadas?
3. ¿Cambia el AUC al recalibrar?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('calibracion', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('calibracion', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El modelo tiene AUC **0,8214** —ordena bien— y un error medio de calibración de **0,0542**: en los tramos altos predice más de lo que ocurre y en los bajos, menos. Tras recalibrar, el Brier baja y el AUC apenas se mueve: la calibración es monótona, **reescala pero no reordena**.


## 10. Comentario pedagógico

Importa cuando la salida se usa para algo más que ordenar: decidir con un umbral de coste, combinar la probabilidad con otra, o presentarla a una persona. Un modelo bien ordenado y mal calibrado toma decisiones sistemáticamente sesgadas, y ningún ranking lo delata.


## 11. Error o anti-patrón deliberado

Anti-patrón: calibrar con los mismos datos con los que se evalúa la calibración.


In [ ]:
print('La calibracion se AJUSTA a unos datos: si evaluas sobre esos mismos,')
print('el error de calibracion sale casi cero por construccion.')
print('Hace falta un conjunto aparte, igual que para cualquier otro ajuste.')

## 12. Corrección

Lo que hay que reportar, y en qué conjunto:


In [ ]:
r = run_paper_lab('calibracion', seed=7)['result']
print('AUC antes / despues :', r['auc_del_modelo'], '/', r['auc_tras_calibrar'])
print('Brier antes / despues:', r['brier_antes'], '/', r['brier_despues'])
for fila in r['diagrama_de_fiabilidad_antes']:
    print(f"  {fila['intervalo']}  predicho={fila['prob_media_predicha']:<8}"
          f" observado={fila['frecuencia_observada']:<8} desv={fila['desviacion']}")

## 13. Desafío guiado

Localiza en el diagrama de fiabilidad el tramo con mayor desviación y di en qué dirección se equivoca el modelo allí.


In [ ]:
r = run_paper_lab('calibracion', seed=3)['result']
show(r)

## 14. Desafío autónomo

Calibra un modelo tuyo con escalado de Platt y con regresión isotónica, usando un conjunto reservado. Compara AUC y Brier antes y después, y decide cuál usarías según el tamaño de los datos.


## 15. Evidencia de aprendizaje

Guarda el diagrama de fiabilidad antes y después con los dos números —AUC y Brier— y tu criterio de cuándo la calibración importa.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P82_calibracion/README.md) · evaluación formal: [`assessments/papers/P82_calibracion.md`](../../assessments/papers/P82_calibracion.md)


## 16. Cierre

Ya se mide bien lo que el modelo predice. Volvemos a mirar los datos: cómo verlos cuando tienen demasiadas dimensiones para dibujarlos.


## 17. Conexión con el siguiente hito

- P50

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
